In [4]:
import json
from pathlib import Path
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.agent import run_agent_with_trace


test_set = json.loads((PROJECT_ROOT / "data" / "test_set.json").read_text())

results = []
for q in test_set:
    trace = run_agent_with_trace(q["question"])
    
    used_tool = len(trace["tool_calls"]) > 0
    correct_tool_use = (used_tool == q["should_use_tool"])
    
    if used_tool and q["expected_tool"]:
        called_expected = any(c["name"] == q["expected_tool"] for c in trace["tool_calls"])
    else:
        called_expected = not used_tool
    
    topics_found = sum(
        1 for t in q["expected_topics"] 
        if t.lower() in (trace["final_answer"] or "").lower()
    )
    topic_coverage = topics_found / max(len(q["expected_topics"]), 1)
    
    results.append({
        "id": q["id"],
        "correct_tool_use": correct_tool_use,
        "called_expected_tool": called_expected,
        "topic_coverage": topic_coverage,
        "n_tool_calls": len(trace["tool_calls"]),
        "final_answer": trace["final_answer"],
    })

# Summary
print(f"Tool use decisions correct: {sum(r['correct_tool_use'] for r in results)}/{len(results)}")
print(f"Correct tool called:        {sum(r['called_expected_tool'] for r in results)}/{len(results)}")
print(f"Avg topic coverage:         {sum(r['topic_coverage'] for r in results) / len(results):.2f}")

Tool use decisions correct: 3/4
Correct tool called:        4/4
Avg topic coverage:         0.42


In [5]:
for r in results:
    if not r["correct_tool_use"]:
        print(f"FAILED: {r['id']}")
        # Find the original test question
        original = next(q for q in test_set if q["id"] == r["id"])
        print(f"  Question: {original['question']}")
        print(f"  Should use tool? {original['should_use_tool']}")
        print(f"  Tool calls made: {r['n_tool_calls']}")
        print(f"  Final answer: {r['final_answer'][:300]}")

FAILED: q4
  Question: What is the Riemann Hypothesis?
  Should use tool? True
  Tool calls made: 0
  Final answer: The Riemann Hypothesis is one of the most famous unsolved problems in mathematics. It concerns the distribution of prime numbers and was formulated by the German mathematician Bernhard Riemann in 1859.

**The Hypothesis:** The Riemann Hypothesis states that all non-trivial zeros of the Riemann zeta 


In [ ]:
"""Inspect each result side-by-side with expectations."""
for r in results:
    original = next(q for q in test_set if q["id"] == r["id"])
    print(f"\n{'='*70}")
    print(f"[{r['id']}] {original['question']}")
    print(f"{'='*70}")
    print(f"Expected: use_tool={original['should_use_tool']}, tool={original['expected_tool']}")
    print(f"Expected keywords: {original['expected_topics']}")
    print(f"Actual: n_tool_calls={r['n_tool_calls']}")
    print(f"Topic coverage: {r['topic_coverage']:.2f}")
    
    # Which keywords were missing?
    answer_lower = (r['final_answer'] or '').lower()
    hit = [t for t in original['expected_topics'] if t.lower() in answer_lower]
    missed = [t for t in original['expected_topics'] if t.lower() not in answer_lower]
    print(f"Hit: {hit}")
    print(f"Missed: {missed}")
    
    print(f"\nAnswer:\n{r['final_answer'][:500]}")


[q1] What is retrieval augmented generation according to the book?
Expected: use_tool=True, tool=search_book
Expected keywords: ['retrieval', 'context', 'generation']
Actual: n_tool_calls=1
Topic coverage: 0.67
Hit: ['retrieval', 'generation']
Missed: ['context']

Answer:
According to the book, **retrieval-augmented generation (RAG)** is defined as follows:

RAG is a solution for knowledge-intensive tasks where not all available knowledge can be input directly into the model. The key idea is that instead of trying to fit all information into the model, only the information most relevant to the query—as determined by a retriever—is retrieved and input into the model.

The term was coined in the paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Task

[q2] What topics does this book cover?
Expected: use_tool=True, tool=list_topics
Expected keywords: ['fine-tuning', 'RAG', 'evaluation']
Actual: n_tool_calls=1
Topic coverage: 1.00
Hit: ['fine-tuning', 'RAG', 'evaluation']
M

: 